# Combined Group Stock-Market Preprocessing Pipeline

This notebook integrates the six members' preprocessing techniques into one leakage-safe workflow.

| Member | Technique |
|---|---|
| IT25102978 | Missing-data handling, validation, target creation and chronological splitting |
| IT25102165 | Categorical encoding of `Index` |
| IT25200808 | IQR-based outlier handling |
| IT25101173 | Standard feature scaling |
| IT25200151 | Calendar feature engineering |
| IT25102034 | SelectKBest feature selection |

Run the notebook from top to bottom. If `data/raw/Market.csv` or `Market.csv` is unavailable, Colab will ask you to upload it.

## 1. Setup

In [1]:
from pathlib import Path
from typing import Iterable
import json
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler

REQUIRED_COLUMNS = ["Index", "Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
OUTLIER_COLUMNS = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
CALENDAR_COLUMNS = ["Year", "Month", "Day", "Day_of_Week"]
MEMBER_TECHNIQUES = {
    "IT25102978": "Missing-data handling and validation",
    "IT25102165": "Categorical encoding",
    "IT25200808": "IQR-based outlier handling",
    "IT25101173": "Standard feature scaling",
    "IT25200151": "Calendar feature engineering",
    "IT25102034": "SelectKBest feature selection",
}
K_FEATURES = 5
print("Setup complete.")

Setup complete.


## 2. Load the raw dataset

In [2]:
def build_output_paths(output_root: Path) -> dict[str, Path]:
    paths = {
        "root": output_root,
        "outputs": output_root / "outputs",
        "charts": output_root / "eda_visualizations",
        "logs": output_root / "logs",
    }
    for directory in paths.values():
        directory.mkdir(parents=True, exist_ok=True)
    return paths


def validate_input_columns(raw_df: pd.DataFrame) -> None:
    missing_columns = [column for column in REQUIRED_COLUMNS if column not in raw_df]
    if missing_columns:
        raise ValueError(f"Input dataset is missing columns: {missing_columns}")

input_candidates = [Path("data/raw/Market.csv"), Path("Market.csv")]
input_path = next((path for path in input_candidates if path.exists()), None)

if input_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        input_path = Path(next(iter(uploaded)))
    except (ImportError, StopIteration):
        raise FileNotFoundError("Place Market.csv in data/raw/ or the notebook directory.")

paths = build_output_paths(Path("results"))
raw_df = pd.read_csv(input_path)
validate_input_columns(raw_df)
print("Loaded:", input_path)
print("Raw shape:", raw_df.shape)
display(raw_df.head())

Loaded: data/raw/Market.csv
Raw shape: (112457, 8)
  Index        Date        Open  ...       Close   Adj Close  Volume
0   NYA  12/31/1965  528.690002  ...  528.690002  528.690002     0.0
1   NYA    1/3/1966  527.210022  ...  527.210022  527.210022     0.0
2   NYA    1/4/1966  527.840027  ...  527.840027  527.840027     0.0
3   NYA    1/5/1966  531.119995  ...  531.119995  531.119995     0.0
4   NYA    1/6/1966  532.070007  ...  532.070007  532.070007     0.0

[5 rows x 8 columns]


## 3. Member 1 — Missing-data handling, validation and target creation

In [3]:
def clean_and_create_target(
    raw_df: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    dict[str, int | float],
]:
    """Member 1: clean raw data and create a next-day classification target."""
    validate_input_columns(raw_df)
    df = raw_df.copy()

    original_rows, original_columns = df.shape
    missing_before = df[REQUIRED_COLUMNS].isna().sum()
    rows_with_missing = int(df[REQUIRED_COLUMNS].isna().any(axis=1).sum())
    duplicate_rows = int(df.duplicated().sum())

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.drop_duplicates()
    df = df.dropna(subset=REQUIRED_COLUMNS).copy()
    rows_after_missing = len(df)

    invalid_mask = (
        (df["Open"] <= 0)
        | (df["High"] <= 0)
        | (df["Low"] <= 0)
        | (df["Close"] <= 0)
        | (df["Adj Close"] <= 0)
        | (df["Volume"] < 0)
        | (df["High"] < df["Low"])
        | (df["High"] < df[["Open", "Close"]].max(axis=1))
        | (df["Low"] > df[["Open", "Close"]].min(axis=1))
    )

    invalid_records = df.loc[invalid_mask].copy()
    df = df.loc[~invalid_mask].copy()
    rows_after_invalid = len(df)

    # Zero volume is retained because some indices do not report volume.
    df["Volume_Available"] = (df["Volume"] > 0).astype(int)
    df = df.sort_values(["Index", "Date"]).reset_index(drop=True)

    # Target creation is performed within each index to prevent cross-index mixing.
    df["Next_Close"] = df.groupby("Index")["Close"].shift(-1)
    df["Target_Date"] = df.groupby("Index")["Date"].shift(-1)
    df = df.dropna(subset=["Next_Close", "Target_Date"]).copy()
    df["Target"] = (df["Next_Close"] > df["Close"]).astype(int)

    cleaned_df = df.drop(columns=["Next_Close", "Target_Date"]).copy()
    missing_after = cleaned_df.isna().sum()

    missing_summary = pd.DataFrame(
        {
            "Column": REQUIRED_COLUMNS + ["Volume_Available", "Target"],
            "Missing_Before": [
                int(missing_before.get(column, 0)) for column in REQUIRED_COLUMNS
            ]
            + [0, 0],
            "Missing_After": [
                int(missing_after.get(column, 0)) for column in REQUIRED_COLUMNS
            ]
            + [
                int(missing_after.get("Volume_Available", 0)),
                int(missing_after.get("Target", 0)),
            ],
        }
    )

    summary: dict[str, int | float] = {
        "original_rows": int(original_rows),
        "original_columns": int(original_columns),
        "market_indices": int(raw_df["Index"].nunique(dropna=True)),
        "rows_with_missing": rows_with_missing,
        "missing_percentage": float(rows_with_missing / original_rows * 100),
        "duplicate_rows": duplicate_rows,
        "rows_after_missing_and_duplicates": int(rows_after_missing),
        "invalid_records_removed": int(invalid_mask.sum()),
        "rows_after_invalid_removal": int(rows_after_invalid),
        "zero_volume_rows": int((cleaned_df["Volume"] == 0).sum()),
        "final_cleaned_rows": int(len(cleaned_df)),
        "final_cleaned_columns": int(cleaned_df.shape[1]),
        "target_down_rows": int((cleaned_df["Target"] == 0).sum()),
        "target_up_rows": int((cleaned_df["Target"] == 1).sum()),
    }

    return cleaned_df, df, invalid_records, missing_summary, summary

In [4]:
cleaned_df, working_df, invalid_records, missing_summary, cleaning_summary = clean_and_create_target(raw_df)
print("Cleaned shape:", cleaned_df.shape)
print("Rows containing missing values:", cleaning_summary["rows_with_missing"])
print("Invalid OHLC rows removed:", cleaning_summary["invalid_records_removed"])
print("Target distribution:")
display(cleaned_df["Target"].value_counts().rename_axis("Target").to_frame("Count"))

Cleaned shape: (110223, 10)
Rows containing missing values: 2219
Invalid OHLC rows removed: 1
Target distribution:
        Count
Target       
1       58508
0       51715


## 4. Member 5 — Calendar feature engineering and chronological splitting

In [5]:
def engineer_calendar_features(dataset: pd.DataFrame) -> pd.DataFrame:
    """Member 5: extract calendar fields from Date."""
    engineered = dataset.copy()
    engineered["Date"] = pd.to_datetime(engineered["Date"], errors="raise")
    engineered["Year"] = engineered["Date"].dt.year
    engineered["Month"] = engineered["Date"].dt.month
    engineered["Day"] = engineered["Date"].dt.day
    engineered["Day_of_Week"] = engineered["Date"].dt.dayofweek
    return engineered


def split_each_index(
    working_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Create chronological 70/15/15 partitions separately for each index."""
    train_parts: list[pd.DataFrame] = []
    validation_parts: list[pd.DataFrame] = []
    test_parts: list[pd.DataFrame] = []
    split_rows: list[dict[str, int | str]] = []

    for index_name, index_data in working_df.groupby("Index", sort=False):
        index_data = index_data.sort_values("Target_Date").copy()
        total = len(index_data)
        train_end = int(total * 0.70)
        validation_end = int(total * 0.85)

        train_part = index_data.iloc[:train_end].copy()
        validation_part = index_data.iloc[train_end:validation_end].copy()
        test_part = index_data.iloc[validation_end:].copy()

        if min(len(train_part), len(validation_part), len(test_part)) == 0:
            raise ValueError(
                f"Index {index_name!r} does not contain enough rows for all splits."
            )

        assert (
            train_part["Target_Date"].max()
            < validation_part["Target_Date"].min()
        )
        assert (
            validation_part["Target_Date"].max()
            < test_part["Target_Date"].min()
        )

        train_parts.append(train_part)
        validation_parts.append(validation_part)
        test_parts.append(test_part)
        split_rows.append(
            {
                "Index": str(index_name),
                "Total": total,
                "Training": len(train_part),
                "Validation": len(validation_part),
                "Testing": len(test_part),
            }
        )

    train = pd.concat(train_parts, ignore_index=True)
    validation = pd.concat(validation_parts, ignore_index=True)
    test = pd.concat(test_parts, ignore_index=True)
    split_summary = pd.DataFrame(split_rows)
    return train, validation, test, split_summary


def remove_target_leakage_columns(dataset: pd.DataFrame) -> pd.DataFrame:
    return dataset.drop(columns=["Next_Close", "Target_Date"]).reset_index(drop=True)

In [6]:
engineered_cleaned_df = engineer_calendar_features(cleaned_df)
engineered_working_df = engineer_calendar_features(working_df)
train_source, validation_source, test_source, split_summary = split_each_index(engineered_working_df)
train_source = remove_target_leakage_columns(train_source)
validation_source = remove_target_leakage_columns(validation_source)
test_source = remove_target_leakage_columns(test_source)
print("Split sizes:", len(train_source), len(validation_source), len(test_source))
display(split_summary.head())

Split sizes: 77150 16534 16539
       Index  Total  Training  Validation  Testing
0  000001.SS   5790      4052         869      869
1  399001.SZ   5759      4031         864      864
2      GDAXI   8437      5905        1266     1266
3     GSPTSE  10525      7367        1579     1579
4        HSI   8491      5943        1274     1274


## 5. Member 3 — IQR-based outlier handling

IQR limits are learned only from training data, then applied unchanged to validation and test data.

In [7]:
def fit_iqr_limits(training_df: pd.DataFrame) -> pd.DataFrame:
    """Member 3: learn per-index IQR limits from training data only."""
    rows: list[dict[str, str | float]] = []

    for index_name, index_data in training_df.groupby("Index", sort=False):
        for column in OUTLIER_COLUMNS:
            values = index_data[column].dropna()

            # Preserve legitimate zero-volume rows.  Bounds are estimated from
            # reported positive volume whenever positive values exist.
            if column == "Volume" and (values > 0).any():
                values = values[values > 0]

            q1 = float(values.quantile(0.25))
            q3 = float(values.quantile(0.75))
            iqr = q3 - q1

            if not np.isfinite(iqr) or iqr <= 0:
                lower = float(values.min())
                upper = float(values.max())
            else:
                lower = q1 - 1.5 * iqr
                upper = q3 + 1.5 * iqr

            if column in {"Open", "High", "Low", "Close", "Adj Close", "Volume"}:
                lower = max(0.0, lower)

            rows.append(
                {
                    "Index": str(index_name),
                    "Feature": column,
                    "Q1": q1,
                    "Q3": q3,
                    "IQR": iqr,
                    "Lower_Bound": lower,
                    "Upper_Bound": upper,
                }
            )

    return pd.DataFrame(rows)


def apply_iqr_limits(
    dataset: pd.DataFrame,
    limits: pd.DataFrame,
) -> tuple[pd.DataFrame, int]:
    capped = dataset.copy()
    changed_values = 0

    for limit in limits.itertuples(index=False):
        index_mask = capped["Index"].astype(str) == str(limit.Index)
        if limit.Feature == "Volume":
            # Zero is a meaningful "not reported" value, not an outlier.
            apply_mask = index_mask & (capped[limit.Feature] > 0)
        else:
            apply_mask = index_mask

        before = capped.loc[apply_mask, limit.Feature].copy()
        capped.loc[apply_mask, limit.Feature] = before.clip(
            lower=float(limit.Lower_Bound),
            upper=float(limit.Upper_Bound),
        )
        after = capped.loc[apply_mask, limit.Feature]
        changed_values += int((before.to_numpy() != after.to_numpy()).sum())

    # Independent capping can slightly disturb OHLC ordering. Restore the
    # financial relationship High >= Open/Close and Low <= Open/Close.
    capped["High"] = capped[["High", "Open", "Close"]].max(axis=1)
    capped["Low"] = capped[["Low", "Open", "Close"]].min(axis=1)
    return capped, changed_values

In [8]:
outlier_limits = fit_iqr_limits(train_source)
train_capped, train_values_capped = apply_iqr_limits(train_source, outlier_limits)
validation_capped, validation_values_capped = apply_iqr_limits(validation_source, outlier_limits)
test_capped, test_values_capped = apply_iqr_limits(test_source, outlier_limits)
print("Values capped — train/validation/test:", train_values_capped, validation_values_capped, test_values_capped)
display(outlier_limits)

Values capped — train/validation/test: 8886 17009 41648
        Index    Feature            Q1  ...           IQR  Lower_Bound   Upper_Bound
0   000001.SS       Open  1.423240e+03  ...  1.014323e+03     0.000000  3.959046e+03
1   000001.SS       High  1.433451e+03  ...  1.021113e+03     0.000000  3.986234e+03
2   000001.SS        Low  1.409682e+03  ...  1.009279e+03     0.000000  3.932878e+03
3   000001.SS      Close  1.422241e+03  ...  1.017262e+03     0.000000  3.965396e+03
4   000001.SS  Adj Close  1.422241e+03  ...  1.017262e+03     0.000000  3.965396e+03
..        ...        ...           ...  ...           ...          ...           ...
79       TWII       High  5.965295e+03  ...  2.143700e+03  2749.744996  1.132454e+04
80       TWII        Low  5.879535e+03  ...  2.128850e+03  2686.259766  1.120166e+04
81       TWII      Close  5.917625e+03  ...  2.137955e+03  2710.692383  1.126251e+04
82       TWII  Adj Close  5.917603e+03  ...  2.137946e+03  2710.683594  1.126247e+04
83       

## 6. IT25102165 — Categorical encoding

In [9]:
def one_hot_encode_index(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """IT25102165: encode Index and align validation/test to training columns."""
    train_encoded = pd.get_dummies(
        train, columns=["Index"], prefix="Index", dtype=int
    )
    validation_encoded = pd.get_dummies(
        validation, columns=["Index"], prefix="Index", dtype=int
    )
    test_encoded = pd.get_dummies(
        test, columns=["Index"], prefix="Index", dtype=int
    )

    validation_encoded = validation_encoded.reindex(
        columns=train_encoded.columns, fill_value=0
    )
    test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

    index_columns = [
        column for column in train_encoded.columns if column.startswith("Index_")
    ]

    for encoded in (train_encoded, validation_encoded, test_encoded):
        active_categories = encoded[index_columns].sum(axis=1)
        assert (active_categories == 1).all()

    return train_encoded, validation_encoded, test_encoded, index_columns


def create_model_matrices(
    train_encoded: pd.DataFrame,
    validation_encoded: pd.DataFrame,
    test_encoded: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.Series,
    pd.Series,
    pd.Series,
]:
    def separate(dataset: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
        target = dataset["Target"].astype(int).reset_index(drop=True)
        features = dataset.drop(columns=["Target", "Date"]).reset_index(drop=True)
        return features, target

    x_train, y_train = separate(train_encoded)
    x_validation, y_validation = separate(validation_encoded)
    x_test, y_test = separate(test_encoded)
    return x_train, x_validation, x_test, y_train, y_validation, y_test

In [10]:
train_encoded, validation_encoded, test_encoded, index_columns = one_hot_encode_index(
    train_capped, validation_capped, test_capped
)
x_train, x_validation, x_test, y_train, y_validation, y_test = create_model_matrices(
    train_encoded, validation_encoded, test_encoded
)
print("One-hot Index columns:", len(index_columns))
print(index_columns)

One-hot Index columns: 14
['Index_000001.SS', 'Index_399001.SZ', 'Index_GDAXI', 'Index_GSPTSE', 'Index_HSI', 'Index_IXIC', 'Index_J203.JO', 'Index_KS11', 'Index_N100', 'Index_N225', 'Index_NSEI', 'Index_NYA', 'Index_SSMI', 'Index_TWII']


## 7. Member 4 — Standard feature scaling

The scaler is fitted only on training data.

In [11]:
def standardize_features(
    x_train: pd.DataFrame,
    x_validation: pd.DataFrame,
    x_test: pd.DataFrame,
    index_columns: Iterable[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Member 4: fit StandardScaler only on training observations."""
    excluded_binary = {"Volume_Available", *index_columns}
    scale_columns = [
        column for column in x_train.columns if column not in excluded_binary
    ]

    scaler = StandardScaler()
    scaler.fit(x_train[scale_columns])

    def transform(dataset: pd.DataFrame) -> pd.DataFrame:
        transformed = dataset.copy()
        transformed = transformed.astype(
            {column: "float64" for column in scale_columns}
        )
        transformed.loc[:, scale_columns] = scaler.transform(
            dataset[scale_columns]
        )
        return transformed

    train_scaled = transform(x_train)
    validation_scaled = transform(x_validation)
    test_scaled = transform(x_test)

    scaler_parameters = pd.DataFrame(
        {
            "Feature": scale_columns,
            "Training_Mean": scaler.mean_,
            "Training_Scale": scaler.scale_,
        }
    )
    return train_scaled, validation_scaled, test_scaled, scaler_parameters

In [12]:
x_train_scaled, x_validation_scaled, x_test_scaled, scaler_parameters = standardize_features(
    x_train, x_validation, x_test, index_columns
)
print("Scaled numerical features:", scaler_parameters["Feature"].tolist())
display(scaler_parameters)

Scaled numerical features: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Year', 'Month', 'Day', 'Day_of_Week']
       Feature  Training_Mean  Training_Scale
0         Open   6.062883e+03    8.391145e+03
1         High   6.099880e+03    8.441459e+03
2          Low   6.022609e+03    8.338961e+03
3        Close   6.061897e+03    8.391196e+03
4    Adj Close   6.061713e+03    8.391279e+03
5       Volume   5.810166e+08    2.644458e+09
6         Year   1.996204e+03    1.224305e+01
7        Month   6.532741e+00    3.431467e+00
8          Day   1.582318e+01    8.708353e+00
9  Day_of_Week   2.010071e+00    1.406204e+00


## 8. Member 6 — SelectKBest feature selection

`f_classif` is used because the target is binary. The selector is fitted only on training data.

In [13]:
def select_best_features(
    x_train: pd.DataFrame,
    x_validation: pd.DataFrame,
    x_test: pd.DataFrame,
    y_train: pd.Series,
    requested_k: int,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    list[str],
]:
    """Member 6: select the strongest classification features on training only."""
    if requested_k < 1:
        raise ValueError("--k-features must be at least 1.")

    k = min(requested_k, x_train.shape[1])
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(x_train, y_train)

    selected_features = x_train.columns[selector.get_support()].tolist()
    score_table = pd.DataFrame(
        {
            "Feature": x_train.columns,
            "F_Score": selector.scores_,
            "P_Value": selector.pvalues_,
            "Selected": selector.get_support(),
        }
    ).sort_values("F_Score", ascending=False, na_position="last")

    return (
        x_train[selected_features].copy(),
        x_validation[selected_features].copy(),
        x_test[selected_features].copy(),
        score_table,
        selected_features,
    )


def add_target(features: pd.DataFrame, target: pd.Series) -> pd.DataFrame:
    output = features.reset_index(drop=True).copy()
    output["Target"] = target.reset_index(drop=True).astype(int)
    return output

In [14]:
x_train_selected, x_validation_selected, x_test_selected, feature_scores, selected_features = select_best_features(
    x_train_scaled, x_validation_scaled, x_test_scaled, y_train, K_FEATURES
)
train_processed = add_target(x_train_scaled, y_train)
validation_processed = add_target(x_validation_scaled, y_validation)
test_processed = add_target(x_test_scaled, y_test)
train_selected = add_target(x_train_selected, y_train)
validation_selected = add_target(x_validation_selected, y_validation)
test_selected = add_target(x_test_selected, y_test)
print("Selected features:", selected_features)
display(feature_scores.head(10))

Selected features: ['Volume_Available', 'Day', 'Index_399001.SZ', 'Index_IXIC', 'Index_TWII']
             Feature    F_Score       P_Value  Selected
16        Index_IXIC  42.011222  9.129602e-11      True
9                Day  14.685516  1.271193e-04      True
12   Index_399001.SZ   9.671010  1.872665e-03      True
6   Volume_Available   8.992051  2.712430e-03      True
24        Index_TWII   6.773950  9.251557e-03      True
0               Open   5.800145  1.602719e-02     False
1               High   5.799203  1.603577e-02     False
4          Adj Close   5.772736  1.627902e-02     False
2                Low   5.770927  1.629578e-02     False
3              Close   5.760292  1.639470e-02     False


## 9. Group EDA visualizations

In [15]:
def chart_missing_values(missing_summary: pd.DataFrame, chart_dir: Path) -> None:
    chart_data = missing_summary.iloc[: len(REQUIRED_COLUMNS)]
    y_positions = np.arange(len(chart_data))
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        y_positions,
        chart_data["Missing_Before"],
        color="#D95F59",
        edgecolor="#7D2E2A",
    )
    ax.set_yticks(y_positions, chart_data["Column"])
    ax.invert_yaxis()
    ax.set_title("Missing Values Before Data Cleaning")
    ax.set_xlabel("Number of missing values")
    ax.set_ylabel("Dataset column")
    maximum = max(int(chart_data["Missing_Before"].max()), 1)
    ax.set_xlim(0, maximum * 1.15)
    ax.grid(axis="x", alpha=0.2)
    for bar in bars:
        value = int(bar.get_width())
        if value:
            ax.text(
                value + maximum * 0.012,
                bar.get_y() + bar.get_height() / 2,
                f"{value:,}",
                va="center",
                fontsize=9,
            )
    fig.tight_layout()
    fig.savefig(
        chart_dir / "01_IT25102978_missing_values.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_target_distribution(cleaned_df: pd.DataFrame, chart_dir: Path) -> None:
    counts = cleaned_df["Target"].value_counts().reindex([0, 1], fill_value=0)
    labels = ["Down (0)", "Up (1)"]
    fig, ax = plt.subplots(figsize=(8, 5.5))
    bars = ax.bar(
        labels,
        counts.values,
        color=["#F2A541", "#3A9D5D"],
        edgecolor="#333333",
    )
    total = int(counts.sum())
    ax.set_title("Next-Day Market Trend Distribution")
    ax.set_xlabel("Target class")
    ax.set_ylabel("Number of observations")
    ax.set_ylim(0, max(counts.max() * 1.16, 1))
    for bar, value in zip(bars, counts.values):
        percentage = value / total * 100 if total else 0
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + total * 0.008,
            f"{value:,}\n({percentage:.2f}%)",
            ha="center",
        )
    fig.tight_layout()
    fig.savefig(
        chart_dir / "07_target_distribution.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_categorical_encoding(
    original_train: pd.DataFrame,
    encoded_train: pd.DataFrame,
    index_columns: list[str],
    chart_dir: Path,
) -> None:
    category_counts = original_train["Index"].value_counts().sort_values(
        ascending=False
    )
    rows_per_index = 4
    ordered_indices = sorted(original_train["Index"].astype(str).unique())
    sample_positions: list[int] = []
    for index_name in ordered_indices:
        positions = original_train.index[
            original_train["Index"].astype(str) == index_name
        ][:rows_per_index]
        sample_positions.extend(positions.tolist())

    encoded_sample = encoded_train.loc[sample_positions, index_columns].T

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].bar(
        category_counts.index,
        category_counts.values,
        color="#2E86AB",
        edgecolor="black",
    )
    axes[0].set_title("Training Records by Market Index")
    axes[0].set_xlabel("Market index")
    axes[0].set_ylabel("Number of records")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", alpha=0.3)

    image = axes[1].imshow(
        encoded_sample.values,
        aspect="auto",
        interpolation="nearest",
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    axes[1].set_title("One-Hot Encoding: Four Rows per Market Index")
    axes[1].set_xlabel("Training row")
    axes[1].set_ylabel("Encoded Index feature")
    axes[1].set_yticks(range(len(index_columns)))
    axes[1].set_yticklabels(index_columns, fontsize=8)
    tick_positions = [
        position * rows_per_index + (rows_per_index - 1) / 2
        for position in range(len(ordered_indices))
    ]
    axes[1].set_xticks(tick_positions)
    axes[1].set_xticklabels(ordered_indices, rotation=60, ha="right", fontsize=7)
    fig.colorbar(image, ax=axes[1], ticks=[0, 1], label="Encoded value")

    fig.suptitle("IT25102165 - Categorical Encoding")
    fig.tight_layout()
    fig.savefig(
        chart_dir / "02_IT25102165_categorical_encoding.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_outlier_handling(
    train_before: pd.DataFrame,
    train_after: pd.DataFrame,
    chart_dir: Path,
) -> None:
    change_counts = pd.Series(
        {
            column: int(
                (train_before[column].to_numpy() != train_after[column].to_numpy()).sum()
            )
            for column in OUTLIER_COLUMNS
        }
    )

    close_changed = train_before["Close"].to_numpy() != train_after["Close"].to_numpy()
    changes_by_index = (
        pd.Series(close_changed, index=train_before.index)
        .groupby(train_before["Index"])
        .sum()
    )
    display_index = str(changes_by_index.idxmax())
    index_mask = train_before["Index"].astype(str) == display_index
    close_before = np.sort(train_before.loc[index_mask, "Close"].to_numpy())
    close_after = np.sort(train_after.loc[index_mask, "Close"].to_numpy())
    percentiles = np.linspace(0, 100, len(close_before))

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].bar(
        change_counts.index,
        change_counts.values,
        color="#E15759",
        edgecolor="#7D2E2A",
    )
    axes[0].set_title("Values Changed by IQR Capping")
    axes[0].set_xlabel("Feature")
    axes[0].set_ylabel("Number of capped training values")
    axes[0].tick_params(axis="x", rotation=35)
    axes[0].grid(axis="y", alpha=0.2)

    axes[1].plot(percentiles, close_before, label="Before", color="#4E79A7")
    axes[1].plot(percentiles, close_after, label="After", color="#E15759")
    axes[1].set_title(f"Close-Price Tail Before/After Capping: {display_index}")
    axes[1].set_xlabel("Empirical percentile")
    axes[1].set_ylabel("Closing price")
    axes[1].set_yscale("log")
    axes[1].grid(alpha=0.2)
    axes[1].legend(frameon=False)
    fig.suptitle("IT25200808 - Outlier Handling")
    fig.tight_layout()
    fig.savefig(
        chart_dir / "03_IT25200808_outlier_handling.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_feature_scaling(
    train_before: pd.DataFrame,
    train_after: pd.DataFrame,
    scaler_parameters: pd.DataFrame,
    chart_dir: Path,
) -> None:
    display_columns = [
        column
        for column in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
        if column in scaler_parameters["Feature"].tolist()
    ]
    before_values = [train_before[column].to_numpy() for column in display_columns]
    after_values = [train_after[column].to_numpy() for column in display_columns]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].boxplot(before_values, tick_labels=display_columns, showfliers=False)
    axes[0].set_title("Before Standard Scaling")
    axes[0].set_yscale("symlog")
    axes[0].set_ylabel("Original feature value (symlog scale)")
    axes[1].boxplot(after_values, tick_labels=display_columns, showfliers=False)
    axes[1].set_title("After Training-Fitted Standard Scaling")
    axes[1].set_ylabel("Standardized feature value")
    for axis in axes:
        axis.tick_params(axis="x", rotation=35)
        axis.grid(axis="y", alpha=0.2)
    fig.suptitle("IT25101173 - Feature Scaling")
    fig.tight_layout()
    fig.savefig(
        chart_dir / "04_IT25101173_feature_scaling.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_feature_engineering(
    engineered_df: pd.DataFrame,
    chart_dir: Path,
) -> None:
    day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    target_rate = (
        engineered_df.groupby("Day_of_Week")["Target"]
        .mean()
        .reindex(range(7))
        .fillna(0)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    axes[0].hist(
        engineered_df["Close"],
        bins=50,
        color="#59A14F",
        edgecolor="white",
    )
    axes[0].set_title("Distribution of Closing Price")
    axes[0].set_xlabel("Closing price")
    axes[0].set_ylabel("Frequency")

    axes[1].bar(day_names, target_rate.values * 100, color="#9C755F")
    axes[1].set_title("Next-Day Up Rate by Engineered Day Feature")
    axes[1].set_xlabel("Day of week")
    axes[1].set_ylabel("Up observations (%)")
    axes[1].set_ylim(0, max(100, float(target_rate.max() * 115)))
    axes[1].grid(axis="y", alpha=0.2)

    fig.suptitle("IT25200151 - Feature Engineering")
    fig.tight_layout()
    fig.savefig(
        chart_dir / "05_IT25200151_feature_engineering.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)


def chart_feature_selection(score_table: pd.DataFrame, chart_dir: Path) -> None:
    chart_data = score_table.dropna(subset=["F_Score"]).head(15).sort_values(
        "F_Score"
    )
    colors = ["#E15759" if selected else "#76B7B2" for selected in chart_data["Selected"]]
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(chart_data["Feature"], chart_data["F_Score"], color=colors)
    ax.set_title("IT25102034 - SelectKBest Classification Scores")
    ax.set_xlabel("ANOVA F-score calculated from training data")
    ax.set_ylabel("Feature")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(
        chart_dir / "06_IT25102034_feature_selection.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

In [16]:
chart_missing_values(missing_summary, paths["charts"])
chart_categorical_encoding(train_capped, train_encoded, index_columns, paths["charts"])
chart_outlier_handling(train_source, train_capped, paths["charts"])
chart_feature_scaling(x_train, x_train_scaled, scaler_parameters, paths["charts"])
chart_feature_engineering(engineered_cleaned_df, paths["charts"])
chart_feature_selection(feature_scores, paths["charts"])
chart_target_distribution(cleaned_df, paths["charts"])

chart_files = sorted(paths["charts"].glob("*.png"))
print("Charts created:", len(chart_files))
for chart_file in chart_files:
    print("-", chart_file)

Charts created: 14
- results/eda_visualizations/01_IT25102978_missing_values.png
- results/eda_visualizations/01_missing_values.png
- results/eda_visualizations/02_IT25102165_categorical_encoding.png
- results/eda_visualizations/02_records_per_index.png
- results/eda_visualizations/03_IT25200808_outlier_handling.png
- results/eda_visualizations/03_target_distribution.png
- results/eda_visualizations/04_IT25101173_feature_scaling.png
- results/eda_visualizations/04_daily_return_distribution.png
- results/eda_visualizations/05_IT25200151_feature_engineering.png
- results/eda_visualizations/05_daily_return_boxplot.png
- results/eda_visualizations/06_IT25102034_feature_selection.png
- results/eda_visualizations/06_feature_correlation_heatmap.png
- results/eda_visualizations/07_scaling_comparison.png
- results/eda_visualizations/07_target_distribution.png


## 10. Validation and output files

In [17]:
def validate_pipeline(
    cleaned_df: pd.DataFrame,
    train_source: pd.DataFrame,
    validation_source: pd.DataFrame,
    test_source: pd.DataFrame,
    train_processed: pd.DataFrame,
    validation_processed: pd.DataFrame,
    test_processed: pd.DataFrame,
    train_selected: pd.DataFrame,
    validation_selected: pd.DataFrame,
    test_selected: pd.DataFrame,
    index_columns: list[str],
) -> None:
    assert cleaned_df.isna().sum().sum() == 0
    assert cleaned_df.duplicated().sum() == 0
    assert set(cleaned_df["Target"].unique()) == {0, 1}
    assert "Next_Close" not in cleaned_df.columns
    assert "Target_Date" not in cleaned_df.columns
    assert (cleaned_df["Volume"] < 0).sum() == 0

    assert len(train_source) + len(validation_source) + len(test_source) == len(
        cleaned_df
    )
    assert len(index_columns) == cleaned_df["Index"].nunique()

    for dataset in (train_processed, validation_processed, test_processed):
        assert dataset.isna().sum().sum() == 0
        assert "Date" not in dataset.columns
        assert "Next_Close" not in dataset.columns
        assert "Target_Date" not in dataset.columns

    assert list(train_processed.columns) == list(validation_processed.columns)
    assert list(train_processed.columns) == list(test_processed.columns)
    assert list(train_selected.columns) == list(validation_selected.columns)
    assert list(train_selected.columns) == list(test_selected.columns)


def save_outputs(
    paths: dict[str, Path],
    cleaned_df: pd.DataFrame,
    engineered_df: pd.DataFrame,
    train_processed: pd.DataFrame,
    validation_processed: pd.DataFrame,
    test_processed: pd.DataFrame,
    train_selected: pd.DataFrame,
    validation_selected: pd.DataFrame,
    test_selected: pd.DataFrame,
    invalid_records: pd.DataFrame,
    missing_summary: pd.DataFrame,
    split_summary: pd.DataFrame,
    outlier_limits: pd.DataFrame,
    scaler_parameters: pd.DataFrame,
    feature_scores: pd.DataFrame,
    summary: dict[str, object],
) -> None:
    cleaned_df.to_csv(paths["outputs"] / "cleaned_market_data.csv", index=False)
    engineered_df.to_csv(
        paths["outputs"] / "feature_engineered_market_data.csv", index=False
    )

    train_processed.to_csv(paths["outputs"] / "train_processed.csv", index=False)
    validation_processed.to_csv(
        paths["outputs"] / "validation_processed.csv", index=False
    )
    test_processed.to_csv(paths["outputs"] / "test_processed.csv", index=False)

    train_selected.to_csv(paths["outputs"] / "train_selected.csv", index=False)
    validation_selected.to_csv(
        paths["outputs"] / "validation_selected.csv", index=False
    )
    test_selected.to_csv(paths["outputs"] / "test_selected.csv", index=False)

    invalid_records.to_csv(paths["logs"] / "invalid_records.csv", index=False)
    missing_summary.to_csv(
        paths["logs"] / "missing_values_summary.csv", index=False
    )
    split_summary.to_csv(
        paths["logs"] / "split_summary_by_index.csv", index=False
    )
    outlier_limits.to_csv(paths["logs"] / "outlier_iqr_limits.csv", index=False)
    scaler_parameters.to_csv(
        paths["logs"] / "scaler_training_parameters.csv", index=False
    )
    feature_scores.to_csv(
        paths["logs"] / "feature_selection_scores.csv", index=False
    )

    with (paths["logs"] / "group_pipeline_summary.json").open(
        "w", encoding="utf-8"
    ) as output_file:
        json.dump(summary, output_file, indent=2)

In [18]:
validate_pipeline(
    cleaned_df, train_source, validation_source, test_source,
    train_processed, validation_processed, test_processed,
    train_selected, validation_selected, test_selected, index_columns
)

summary = {
    "member_techniques": MEMBER_TECHNIQUES,
    **cleaning_summary,
    "feature_engineered_columns": CALENDAR_COLUMNS,
    "training_rows": int(len(train_processed)),
    "validation_rows": int(len(validation_processed)),
    "testing_rows": int(len(test_processed)),
    "one_hot_index_columns": index_columns,
    "training_values_iqr_capped": train_values_capped,
    "validation_values_iqr_capped": validation_values_capped,
    "testing_values_iqr_capped": test_values_capped,
    "scaled_features": scaler_parameters["Feature"].tolist(),
    "selected_feature_count": len(selected_features),
    "selected_features": selected_features,
    "full_processed_columns": int(train_processed.shape[1]),
    "selected_output_columns": int(train_selected.shape[1]),
}

save_outputs(
    paths, cleaned_df, engineered_cleaned_df,
    train_processed, validation_processed, test_processed,
    train_selected, validation_selected, test_selected,
    invalid_records, missing_summary, split_summary, outlier_limits,
    scaler_parameters, feature_scores, summary
)

print("ALL GROUP PIPELINE CHECKS PASSED")
print("Cleaned dataset:", cleaned_df.shape)
print("Train/validation/test:", len(train_processed), len(validation_processed), len(test_processed))
print("Outputs folder:", paths["outputs"])
print("Charts folder:", paths["charts"])
print("Logs folder:", paths["logs"])

ALL GROUP PIPELINE CHECKS PASSED
Cleaned dataset: (110223, 10)
Train/validation/test: 77150 16534 16539
Outputs folder: results/outputs
Charts folder: results/eda_visualizations
Logs folder: results/logs


## 11. Create a downloadable results ZIP

In [19]:
zip_path = Path("AIML_Group_Pipeline_Results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for result_file in sorted(Path("results").rglob("*")):
        if result_file.is_file():
            archive.write(result_file, result_file.as_posix())

print("Created:", zip_path.resolve())
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download the ZIP from the notebook working directory.")

Created: /workspace/scratch/6a4517a9146f/AIML_Group_Progress_Review_I/AIML_Group_Pipeline_Results.zip
Download the ZIP from the notebook working directory.
